In [1]:
import pandas as pd

In [4]:
df = pd.read_parquet('sample10k_results/sample10k_sufficiency_classification_results_2026-02-18.parquet')
df

,domain,subdomain,cluster,justice_considerations,natural_resources,planetary_boundaries,wellbeing,impact_summary,sufficiency_classification_reasoning,sufficiency_classification
0,BUILDING,Building Energy Efficiency and Decarbonization,Building Energy Performance Certification and ...,{'evidences': [{'evidence': 'Building Energy P...,{'evidences': [{'evidence': 'Document 7 discus...,{'evidences': [{'evidence': 'Building Energy P...,{'evidences': [{'evidence': 'Building Energy P...,Domain : BUILDING\nSubdomain : Building Energy...,The policy of Building Energy Performance Cert...,PS
1,BUILDING,Building Energy Efficiency and Decarbonization,Energy Efficiency Standards and Codes for New ...,{'evidences': [{'evidence': 'The paper discuss...,{'evidences': [{'evidence': 'Energy requiremen...,{'evidences': [{'evidence': 'Energy efficiency...,{'evidences': [{'evidence': 'Energy Efficiency...,Domain : BUILDING\nSubdomain : Building Energy...,The policy of Energy Efficiency Standards and ...,PS
2,BUILDING,Building Energy Efficiency and Decarbonization,Thermal and Envelope Retrofit Programs for Exi...,{'evidences': [{'evidence': 'The study identif...,{'evidences': [{'evidence': 'Thermal and envel...,{'evidences': [{'evidence': 'Thermal and envel...,{'evidences': [{'evidence': 'Thermal retrofitt...,Domain : BUILDING\nSubdomain : Building Energy...,The policy of thermal and envelope retrofit pr...,S
3,BUILDING,Building Energy Efficiency and Decarbonization,Operational Energy Optimization and HVAC Contr...,{'evidences': [{'evidence': 'The paper analyze...,{'evidences': [{'evidence': 'Operational Energ...,{'evidences': [{'evidence': 'The documents pro...,{'evidences': [{'evidence': 'Operational Energ...,Domain : BUILDING\nSubdomain : Building Energy...,The policy of Operational Energy Optimization ...,PS
4,BUILDING,Building Energy Efficiency and Decarbonization,Embodied Energy and Life Cycle Assessment (LCA...,{'evidences': [{'evidence': 'Life Cycle Assess...,{'evidences': [{'evidence': 'The use of wood s...,{'evidences': [{'evidence': 'Life Cycle Assess...,{'evidences': [{'evidence': 'The documents dis...,Domain : BUILDING\nSubdomain : Building Energy...,The policy of incorporating Embodied Energy an...,PS
...,...,...,...,...,...,...,...,...,...,...
2180,URBAN,Built Environment Safety and Resilience,"Building safety and resilience (e.g., earthqua...",{'evidences': [{'evidence': 'Building codes an...,"{'evidences': [], 'overall_impact': 'neutral'}",{'evidences': [{'evidence': 'Building safety a...,{'evidences': [{'evidence': 'Earthquake-resist...,Domain : URBAN\nSubdomain : Built Environment ...,The policy of building safety and resilience (...,PS
2181,URBAN,Built Environment Safety and Resilience,"Material and design innovation (e.g., reflecti...",{'evidences': [{'evidence': 'Procedural justic...,{'evidences': [{'evidence': 'Technological inn...,{'evidences': [{'evidence': 'The provided docu...,{'evidences': [{'evidence': 'Natural materials...,Domain : URBAN\nSubdomain : Built Environment ...,The policy of material and design innovation (...,PS
2182,URBAN,Rural Development and Livelihood Support,"Rural infrastructure development (e.g., roads,...",{'evidences': [{'evidence': 'Rural infrastruct...,{'evidences': [{'evidence': 'The degradative i...,{'evidences': [{'evidence': 'The development o...,{'evidences': [{'evidence': 'Rural infrastruct...,Domain : URBAN\nSubdomain : Rural Development ...,The policy of rural infrastructure development...,NS
2183,URBAN,Rural Development and Livelihood Support,Support for aging farmers and flexible employm...,{'evidences': [{'evidence': 'The provided docu...,{'evidences': [{'evidence': 'Support for aging...,{'evidences': [{'evidence': 'Support for aging...,{'evidences': [{'evidence': 'Older farmers are...,Domain : URBAN\nSubdomain : Rural Development ...,The policy of supporting aging farmers and fle...,S


In [6]:
from sentence_transformers import SentenceTransformer

In [7]:
model_name = "Qwen/Qwen3-Embedding-0.6B"
model = SentenceTransformer(model_name)
model.half()

SentenceTransformer(
  (0): Transformer({'max_seq_length': 32768, 'do_lower_case': False, 'architecture': 'Qwen3Model'})
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': True, 'include_prompt': True})
  (2): Normalize()
)

In [9]:
prompt = "Embed this policy for semantic retrieval:"

In [15]:
embeddings = model.encode(
    df["cluster"].values,
    convert_to_tensor=True,
    show_progress_bar=True,
    prompt=prompt,
    batch_size=32,
)

Batches:   0%|          | 0/69 [00:00<?, ?it/s]

In [18]:
embeddings.shape

torch.Size([2180, 1024])

In [ ]:
import os
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

load_dotenv()

True

In [32]:
client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
)
client.get_collections()

CollectionsResponse(collections=[CollectionDescription(name='library-v1')])

In [33]:
collection = "policies-v1"
dim = 1024

In [34]:
client.create_collection(
    collection_name=collection,
    vectors_config=VectorParams(size=dim, distance=Distance.COSINE),
)

True

In [35]:
client.get_collections()

CollectionsResponse(collections=[CollectionDescription(name='library-v1'), CollectionDescription(name='policies-v1')])

In [66]:
sdf = df.drop(columns=['justice_considerations', 'planetary_boundaries', 'natural_resources', 'wellbeing'])

In [70]:
vectors = [v[:dim].numpy() for v in embeddings]
payloads = [row.to_dict() for _, row in sdf.iterrows()]

In [ ]:
client.upload_collection(
    collection,
    vectors=vectors,
    payload=payloads,
    parallel=4,
    batch_size=100
)

In [75]:
query = "We should lower the cost of train"
query_embedding = model.encode(query)
top_k = 10

In [76]:
hits = client.query_points(
    collection_name=collection,
    query=query_embedding,
    limit=top_k,
)

In [81]:
[h.payload for h in hits.points]

[{'domain': 'MOBILITY',
  'subdomain': 'Public Transport Affordability Measures',
  'cluster': 'Public transport fare reductions and free transit programs',
  'impact_summary': 'Domain : MOBILITY\nSubdomain : Public Transport Affordability Measures\nPolicy : Public transport fare reductions and free transit programs\n\nImpact on : planetary boundaries\nOverall evaluation : positive\nEvidences :\n - positive: Public transit is critical to achieving the greenhouse gas emissions reductions in urban areas consistent with keeping global temperature increases at 1.5°C or less. Public transit is also a key component of efforts to reduce air pollution. (W3167242726) \n - positive: Applying only welfare-increasing policies captures most of the emission reductions: overall, they reduce emissions by 20% in 15 years. Welfare increases are significantly associated to emission decreases (p-value lower than 0.1%). (W4309442599) \n - positive: Transportation policies aimed at reducing traffic offer an